# Week 08 Lab — Deep Learning Basics: MLP and CNN (Solutions)

Complete reference solutions.

---
# Part 1 — Simple Multi-Layer Perceptron (MLP) on MNIST

In this part you will build, train, and evaluate a fully-connected neural
network to classify handwritten digits from the MNIST dataset.

**Learning goals**
- Understand how to prepare image data for a dense neural network
- Build a Sequential Keras model with `Dense` layers
- Train and evaluate a model on a real dataset


## Step 1 — Imports

Import all libraries needed for the MLP experiment.


In [ ]:
from __future__ import print_function

import keras
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import RMSprop
import numpy as np
from tensorflow.python.keras.utils import np_utils
import matplotlib.pyplot as plt


## Step 2 — Set Parameters and Load MNIST Data

MNIST contains 70 000 greyscale images of handwritten digits (28×28 pixels).


In [ ]:
batch_size  = 20000
num_classes = 10
epochs      = 5

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("x_train shape:", x_train.shape)          # (60000, 28, 28)
print("Number of 8s in training set:", np.sum(y_train == 8))


## Step 3 — Visualise a Sample

Before preprocessing, inspect a raw image and its label.


In [ ]:
plt.imshow(x_train[3890], cmap=plt.cm.binary)
plt.title("Label: " + str(y_train[3890]))
plt.show()


## Step 4 — Preprocess the Data

Dense layers expect a 1-D feature vector, not a 2-D image, so we must
**flatten** each 28×28 image into a vector of length 784.
We also **normalise** pixel values from [0, 255] to [0, 1].


In [ ]:
x_train = x_train.reshape(60000, 784)
x_test  = x_test.reshape(10000, 784)
x_train = x_train.astype('float32')
x_test  = x_test.astype('float32')
x_train /= 255
x_test  /= 255

print(x_train.shape[0], "train samples")   # 60000
print(x_test.shape[0],  "test samples")    # 10000


## Step 5 — One-Hot Encode the Labels

The output layer will have 10 neurons (one per class).
We convert integer labels (e.g. `5`) into binary vectors
(e.g. `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`).


In [ ]:
# Save originals for display (already converted above in solution flow)
# Note: in the actual solution flow, load y_train before this cell.
print("Before encoding:", y_train[:5])           # integer labels

y_train = np_utils.to_categorical(y_train, num_classes)
y_test  = np_utils.to_categorical(y_test,  num_classes)

print("After encoding, y_train shape:", y_train.shape)  # (60000, 10)


## Step 6 — Build the MLP Model

We use Keras' `Sequential` API to stack layers.
Our simple MLP has:
- **One hidden layer**: 512 neurons, ReLU activation
- **One output layer**: 10 neurons (one per class), Softmax activation

> **Tip**: `model.summary()` prints a table showing every layer,
> its output shape, and the number of trainable parameters.


In [ ]:
model = Sequential()
model.add(Dense(512, activation='relu', input_shape=(784,)))
model.add(Dense(num_classes, activation='softmax'))

model.summary()


## Step 7 — Visualise the Model Architecture

`plot_model` draws a diagram of the layer graph.


In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(model, show_shapes=True, show_layer_names=True)


## Step 8 — Compile the Model

Before training we must choose:
- **Loss function** – `categorical_crossentropy` for multi-class classification
- **Optimizer** – `RMSprop` adapts the learning rate during training
- **Metric** – `accuracy` is easy to interpret


In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer=RMSprop(),
    metrics=['accuracy']
)


## Step 9 — Train the Model

`model.fit` runs the training loop.
The `history` object it returns lets us plot learning curves later.


In [ ]:
history = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=10,
    verbose=1,
    validation_data=(x_test, y_test)
)


## Step 10 — Evaluate the Model

Run the model on the held-out test set and report the final metrics.


In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:    ', score[0])
print('Test accuracy:', score[1])


## Step 11 — Reflection

Answer the following questions in a new markdown cell:

1. How many **trainable parameters** does the MLP have?
   Where does most of the computation happen?
2. What accuracy did you achieve after 10 epochs?
   Do you think the model is overfitting or underfitting?
3. The commented-out lines in the original notebook add a second hidden
   layer and `Dropout`. Try uncommenting them. How does accuracy change?
4. Why do we need `softmax` in the output layer but `relu` in hidden layers?


---
# Part 2 — Simple Convolutional Neural Network (CNN) on MNIST

CNNs learn **spatial features** directly from 2-D images using convolutional
filters instead of flattening the image up-front.

**Learning goals**
- Understand how CNNs differ from MLPs in terms of input shape
- Build a CNN with `Conv2D`, `MaxPooling2D`, and `Dropout` layers
- Compare CNN performance with the MLP from Part 1


## Step 1 — Imports (CNN)

In addition to the MLP imports, we need CNN-specific layers.


In [ ]:
from __future__ import print_function

import tensorflow as tf
import keras
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras import backend as K
from tensorflow.python.keras.utils import np_utils


## Step 2 — Set Parameters and Load Data

CNNs keep the spatial dimensions.
The input shape for MNIST is **(28, 28, 1)** (height × width × channels).

> **channels_first vs channels_last**: Some GPU backends expect the
> channel dimension first `(1, 28, 28)`. We use `K.image_data_format()`
> to handle both cases automatically.


In [ ]:
batch_size  = 128
num_classes = 10
epochs      = 12
img_rows, img_cols = 28, 28

(x_train, y_train), (x_test, y_test) = mnist.load_data()

if K.image_data_format() == 'channels_first':
    x_train     = x_train.reshape(x_train.shape[0], 1, img_rows, img_cols)
    x_test      = x_test.reshape(x_test.shape[0],   1, img_rows, img_cols)
    input_shape = (1, img_rows, img_cols)
else:
    x_train     = x_train.reshape(x_train.shape[0], img_rows, img_cols, 1)
    x_test      = x_test.reshape(x_test.shape[0],   img_rows, img_cols, 1)
    input_shape = (img_rows, img_cols, 1)

x_train = x_train.astype('float32') / 255
x_test  = x_test.astype('float32')  / 255

print('x_train shape:', x_train.shape)   # (60000, 28, 28, 1)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0],  'test samples')

y_train = np_utils.to_categorical(y_train, num_classes)
y_test  = np_utils.to_categorical(y_test,  num_classes)


## Step 3 — Build the CNN Model

The architecture:

| Layer | Output shape | Notes |
|---|---|---|
| Conv2D(32, 3×3, relu) | (26, 26, 32) | Learns 32 local filters |
| Conv2D(64, 3×3, relu) | (24, 24, 64) | Deeper feature maps |
| MaxPooling2D(2×2) | (12, 12, 64) | Down-samples by 2 |
| Dropout(0.25) | (12, 12, 64) | Regularisation |
| Flatten | (9216,) | Converts 3-D → 1-D |
| Dense(128, relu) | (128,) | Fully-connected |
| Dropout(0.5) | (128,) | More regularisation |
| Dense(10, softmax) | (10,) | One probability per class |


In [ ]:
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.summary()


## Step 4 — Visualise and Compile the CNN


In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(model, show_shapes=True, show_layer_names=True)

model.compile(
    loss=keras.losses.categorical_crossentropy,
    optimizer=tf.keras.optimizers.Adadelta(),
    metrics=['accuracy']
)


## Step 5 — Train the CNN

> **Note**: Training a CNN takes longer than an MLP.
> We use only **3 epochs** here for speed — try more when you have time!


In [ ]:
epochs = 3

history = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    verbose=1,
    validation_data=(x_test, y_test)
)


## Step 6 — Evaluate the CNN


In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:    ', score[0])
print('Test accuracy:', score[1])


## Step 7 — Compare MLP vs CNN

Answer these questions in a new markdown cell:

1. How many **trainable parameters** does the CNN have compared to the MLP?
   Which model is more parameter-efficient?
2. After only 3 epochs the CNN accuracy may be lower than the MLP after 10
   epochs. Why might this be, and what would happen with more epochs?
3. What is the role of `MaxPooling2D` and `Dropout` in the CNN?
4. When would you prefer a CNN over an MLP for image classification?
   When might an MLP be sufficient?
